In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 0 — Verify Input Paths

In [1]:
import os, glob

PREP_DIR = '/kaggle/input/datasets/majorproject777/flood-preprocessed'
TEMP_DIR = '/kaggle/input/datasets/majorproject777/preprocessed-temporal/preprocessed_temporal'

TRAIN_SPATIAL_DIR  = f'{PREP_DIR}/train'
VAL_SPATIAL_DIR    = f'{PREP_DIR}/val'
TRAIN_TEMPORAL_DIR = f'{TEMP_DIR}/train'
VAL_TEMPORAL_DIR   = f'{TEMP_DIR}/val'
NORM_STATS_PATH    = f'{PREP_DIR}/norm_stats.json'

checks = {
    'Train spatial dir' : TRAIN_SPATIAL_DIR,
    'Val spatial dir'   : VAL_SPATIAL_DIR,
    'Train temporal dir': TRAIN_TEMPORAL_DIR,
    'Val temporal dir'  : VAL_TEMPORAL_DIR,
    'norm_stats.json'   : NORM_STATS_PATH,
}

all_ok = True
for name, path in checks.items():
    if os.path.isdir(path):
        n = len(glob.glob(f'{path}/*.npy'))
        print(f'  {"✅" if n > 0 else "❌"} {name:25s}: {n} .npy files')
        if n == 0: all_ok = False
    elif os.path.isfile(path):
        print(f'  ✅ {name:25s}: exists')
    else:
        print(f'  ❌ {name:25s}: NOT FOUND at {path}')
        all_ok = False

print()
print('All paths verified ✅' if all_ok else 'Fix missing paths ❌')

  ✅ Train spatial dir        : 4128 .npy files
  ✅ Val spatial dir          : 336 .npy files
  ✅ Train temporal dir       : 2064 .npy files
  ✅ Val temporal dir         : 168 .npy files
  ✅ norm_stats.json          : exists

All paths verified ✅


# 1 — Imports & Configuration

In [2]:
import os, json, glob, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

BATCH_SIZE   = 4
NUM_WORKERS  = 2
PIN_MEMORY   = True
LR           = 1e-4
MAX_EPOCHS   = 100
PATIENCE     = 10
GRAD_CLIP    = 1.0
RANDOM_SEED  = 42

SPATIAL_CHANNELS  = 14
TEMPORAL_CHANNELS = 2
N_TIMESTEPS       = 15
CONVLSTM_HIDDEN   = 256   # increased from 128
CONVLSTM_LAYERS   = 2     # increased from 1
BASE_FEATURES     = 64

CKPT_DIR = '/kaggle/working/unet_convlstm_v2'
os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

print(f'Device          : {DEVICE}')
print(f'GPU             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'ConvLSTM hidden : {CONVLSTM_HIDDEN}')
print(f'ConvLSTM layers : {CONVLSTM_LAYERS}')

Device          : cuda
GPU             : Tesla T4
ConvLSTM hidden : 256
ConvLSTM layers : 2


# 2 — Dataset

In [3]:
class FloodDataset(Dataset):
    def __init__(self, spatial_dir, temporal_dir, augment=False):
        self.spatial_dir  = spatial_dir
        self.temporal_dir = temporal_dir
        self.augment      = augment
        self.chip_ids = sorted([
            os.path.basename(f).replace('_spatial.npy', '')
            for f in glob.glob(f'{spatial_dir}/*_spatial.npy')
        ])
        missing = [
            c for c in self.chip_ids
            if not os.path.exists(f'{temporal_dir}/{c}_temporal.npy')
        ]
        if missing:
            raise FileNotFoundError(f'{len(missing)} temporal files missing. First 3: {missing[:3]}')
        print(f'Dataset: {len(self.chip_ids)} chips | augment={augment}')

    def __len__(self): return len(self.chip_ids)

    def __getitem__(self, idx):
        chip_id  = self.chip_ids[idx]
        spatial  = np.load(f'{self.spatial_dir}/{chip_id}_spatial.npy')
        temporal = np.load(f'{self.temporal_dir}/{chip_id}_temporal.npy')
        label    = np.load(f'{self.spatial_dir}/{chip_id}_label.npy')
        if self.augment:
            spatial, temporal, label = self._augment(spatial, temporal, label)
        return (
            torch.from_numpy(spatial.astype(np.float32)),
            torch.from_numpy(temporal.astype(np.float32)),
            torch.from_numpy(label.astype(np.int16)),
        )

    def _augment(self, spatial, temporal, label):
        if random.random() > 0.5:
            spatial  = spatial[:, :, ::-1].copy()
            temporal = temporal[:, :, :, ::-1].copy()
            label    = label[:, ::-1].copy()
        if random.random() > 0.5:
            spatial  = spatial[:, ::-1, :].copy()
            temporal = temporal[:, :, ::-1, :].copy()
            label    = label[::-1, :].copy()
        k = random.randint(0, 3)
        if k > 0:
            spatial  = np.rot90(spatial,  k, axes=(1, 2)).copy()
            temporal = np.rot90(temporal, k, axes=(2, 3)).copy()
            label    = np.rot90(label,    k, axes=(0, 1)).copy()
        return spatial, temporal, label


train_dataset = FloodDataset(TRAIN_SPATIAL_DIR, TRAIN_TEMPORAL_DIR, augment=True)
val_dataset   = FloodDataset(VAL_SPATIAL_DIR,   VAL_TEMPORAL_DIR,   augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
sp, tp, lb = next(iter(train_loader))
print(f'Spatial NaN  : {torch.isnan(sp).any().item()}')
print(f'Temporal NaN : {torch.isnan(tp).any().item()}')

Dataset: 2064 chips | augment=True
Dataset: 168 chips | augment=False
Train batches: 516
Val batches  : 42
Spatial NaN  : False
Temporal NaN : False


# 3 — Model Architecture

In [4]:
# ── U-Net Blocks ──────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = ConvBlock(in_ch, out_ch)
    def forward(self, x): return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], dim=1))


# ── Temporal Attention ────────────────────────────────────────────────────
class TemporalAttention(nn.Module):
    """
    Lightweight temporal self-attention.
    Learns to weight each of the 15 timesteps by relevance.
    E.g. heavy rainfall 2 days before flood → high weight.
    Light drizzle 14 days prior → low weight.

    Input : (B, T, C, H, W)
    Output: (B, T, C, H, W) — reweighted sequence
    """
    def __init__(self, in_channels, n_timesteps):
        super().__init__()
        # Compute attention score per timestep from spatially pooled features
        self.attn = nn.Sequential(
            nn.Linear(in_channels, in_channels // 2),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // 2, 1),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        # Spatial mean per timestep: (B, T, C)
        x_mean = x.mean(dim=(-2, -1))              # (B, T, C)
        # Attention scores: (B, T, 1)
        scores = self.attn(x_mean)                 # (B, T, 1)
        weights = torch.softmax(scores, dim=1)     # (B, T, 1) — sum to 1 over T
        # Reweight sequence
        weights = weights.unsqueeze(-1).unsqueeze(-1)  # (B, T, 1, 1, 1)
        return x * weights                         # (B, T, C, H, W)


# ── Multi-layer ConvLSTM ──────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        pad = kernel_size // 2
        self.gates = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size, padding=pad, bias=True
        )

    def forward(self, x, h, c):
        i, f, o, g = self.gates(torch.cat([x, h], dim=1)).chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        c = f * c + i * torch.tanh(g)
        h = o * torch.tanh(c)
        return h, c


class ConvLSTMEncoder(nn.Module):
    """
    Multi-layer ConvLSTM encoder with dropout between layers.
    Input : (B, T, C, H, W)
    Output: (B, hidden_channels, H, W) — final hidden state of last layer
    """
    def __init__(self, in_channels, hidden_channels, n_layers=2,
                 kernel_size=3, dropout=0.1):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.n_layers        = n_layers

        # Build n_layers ConvLSTM cells
        # Layer 0: input channels → hidden
        # Layers 1+: hidden → hidden
        self.cells = nn.ModuleList()
        for i in range(n_layers):
            in_ch = in_channels if i == 0 else hidden_channels
            self.cells.append(ConvLSTMCell(in_ch, hidden_channels, kernel_size))

        self.dropout = nn.Dropout2d(p=dropout)

    def forward(self, x):
        B, T, C, H, W = x.shape

        # Initialize hidden and cell states for each layer
        h = [torch.zeros(B, self.hidden_channels, H, W, device=x.device)
             for _ in range(self.n_layers)]
        c = [torch.zeros(B, self.hidden_channels, H, W, device=x.device)
             for _ in range(self.n_layers)]

        for t in range(T):
            inp = x[:, t]          # (B, C, H, W) for layer 0
            for layer_idx, cell in enumerate(self.cells):
                h[layer_idx], c[layer_idx] = cell(inp, h[layer_idx], c[layer_idx])
                inp = self.dropout(h[layer_idx])  # dropout between layers

        return h[-1]  # final hidden state of last layer: (B, hidden_channels, H, W)


# ── Full Model ────────────────────────────────────────────────────────────
class UNetConvLSTMV2(nn.Module):
    """
    U-Net + Improved ConvLSTM (Variant 4).
    Improvements: larger hidden, deeper layers, temporal attention, dropout.
    """
    def __init__(self,
                 spatial_channels  = 14,
                 temporal_channels = 2,
                 n_timesteps       = 15,
                 convlstm_hidden   = 256,
                 convlstm_layers   = 2,
                 base_features     = 64):
        super().__init__()
        f = base_features

        # Spatial encoder
        self.enc1    = ConvBlock(spatial_channels, f)
        self.enc2    = Down(f,   f*2)
        self.enc3    = Down(f*2, f*4)
        self.enc4    = Down(f*4, f*8)
        self.down_bn = nn.MaxPool2d(2)

        # Temporal attention — applied before ConvLSTM
        self.temp_attn = TemporalAttention(temporal_channels, n_timesteps)

        # Downsample temporal to bottleneck resolution (32×32)
        self.temp_pool = nn.AdaptiveAvgPool2d((32, 32))

        # Multi-layer ConvLSTM
        self.convlstm = ConvLSTMEncoder(
            in_channels     = temporal_channels,
            hidden_channels = convlstm_hidden,
            n_layers        = convlstm_layers,
            dropout         = 0.1,
        )

        # Project ConvLSTM output to match spatial bottleneck channels
        self.temp_proj = nn.Sequential(
            nn.Conv2d(convlstm_hidden, f*8, 1, bias=False),
            nn.BatchNorm2d(f*8),
            nn.ReLU(inplace=True),
        )

        # Fusion + bottleneck
        self.fusion_conv = nn.Sequential(
            nn.Conv2d(f*8 + f*8, f*8, 1, bias=False),
            nn.BatchNorm2d(f*8),
            nn.ReLU(inplace=True),
        )
        self.bottleneck = ConvBlock(f*8, f*16)

        # Decoder
        self.dec4 = Up(f*16, f*8,  f*8)
        self.dec3 = Up(f*8,  f*4,  f*4)
        self.dec2 = Up(f*4,  f*2,  f*2)
        self.dec1 = Up(f*2,  f,    f)
        self.head = nn.Conv2d(f, 1, 1)

    def forward(self, spatial, temporal):
        # Spatial encoder
        e1   = self.enc1(spatial)
        e2   = self.enc2(e1)
        e3   = self.enc3(e2)
        e4   = self.enc4(e3)
        s_bn = self.down_bn(e4)              # (B, f*8, 32, 32)

        # Temporal attention — reweight timesteps
        temporal = self.temp_attn(temporal)  # (B, T, C, H, W) reweighted

        # Downsample each timestep to 32×32
        B, T, C, H, W = temporal.shape
        t_flat  = temporal.view(B * T, C, H, W)
        t_small = self.temp_pool(t_flat)     # (B*T, C, 32, 32)
        t_seq   = t_small.view(B, T, C, 32, 32)

        # Multi-layer ConvLSTM
        t_feat = self.convlstm(t_seq)        # (B, hidden, 32, 32)
        t_proj = self.temp_proj(t_feat)      # (B, f*8,   32, 32)

        # Fusion
        fused = torch.cat([s_bn, t_proj], dim=1)
        fused = self.fusion_conv(fused)
        bn    = self.bottleneck(fused)

        # Decoder
        d4 = self.dec4(bn, e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.head(d1)


# Instantiate
model = UNetConvLSTMV2(
    spatial_channels  = SPATIAL_CHANNELS,
    temporal_channels = TEMPORAL_CHANNELS,
    n_timesteps       = N_TIMESTEPS,
    convlstm_hidden   = CONVLSTM_HIDDEN,
    convlstm_layers   = CONVLSTM_LAYERS,
    base_features     = BASE_FEATURES,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params/1e6:.1f}M')

with torch.no_grad():
    dummy_sp  = torch.randn(1, SPATIAL_CHANNELS, 512, 512).to(DEVICE)
    dummy_tp  = torch.randn(1, N_TIMESTEPS, TEMPORAL_CHANNELS, 512, 512).to(DEVICE)
    dummy_out = model(dummy_sp, dummy_tp)
    print(f'Input spatial  : {dummy_sp.shape}')
    print(f'Input temporal : {dummy_tp.shape}')
    print(f'Output         : {dummy_out.shape}  (expected (1, 1, 512, 512))')
del dummy_sp, dummy_tp, dummy_out
torch.cuda.empty_cache()

Parameters: 39.1M
Input spatial  : torch.Size([1, 14, 512, 512])
Input temporal : torch.Size([1, 15, 2, 512, 512])
Output         : torch.Size([1, 1, 512, 512])  (expected (1, 1, 512, 512))


# 4 — Loss & Metrics

In [5]:
def dice_loss(pred, target, mask, eps=1e-6):
    pred   = torch.sigmoid(pred).squeeze(1)
    target = target.float()
    pred   = pred * mask
    target = target * mask
    intersection = (pred * target).sum(dim=(1, 2))
    union        = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    return 1 - ((2 * intersection + eps) / (union + eps)).mean()

def bce_loss(pred, target, mask):
    return F.binary_cross_entropy_with_logits(
        pred.squeeze(1)[mask], target.float()[mask]
    )

def combined_loss(pred, target):
    mask        = (target != -1)
    label_clean = target.clone()
    label_clean[~mask] = 0
    return 0.5 * dice_loss(pred, label_clean, mask) + \
           0.5 * bce_loss(pred,  label_clean, mask)

def compute_metrics(pred_logits, target):
    mask     = (target != -1)
    pred_bin = (torch.sigmoid(pred_logits).squeeze(1) > 0.5).long()
    p = pred_bin[mask].cpu()
    t = target[mask].cpu()
    TP = ((p==1) & (t==1)).sum().float()
    FP = ((p==1) & (t==0)).sum().float()
    FN = ((p==0) & (t==1)).sum().float()
    eps = 1e-6
    return {
        'iou'      : (TP / (TP + FP + FN + eps)).item(),
        'f1'       : (2*TP / (2*TP + FP + FN + eps)).item(),
        'precision': (TP / (TP + FP + eps)).item(),
        'recall'   : (TP / (TP + FN + eps)).item(),
    }

print('Loss and metrics defined ✅')

Loss and metrics defined ✅


# 5 — Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

BEST_MODEL_PATH = f'{CKPT_DIR}/best_model.pth'
LAST_MODEL_PATH = f'{CKPT_DIR}/last_model.pth'
LOG_PATH        = f'{CKPT_DIR}/training_log.json'

start_epoch    = 0
best_val_iou   = 0.0
patience_count = 0
training_log   = []

if os.path.exists(LAST_MODEL_PATH):
    print('Resuming from checkpoint...')
    ckpt = torch.load(LAST_MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch    = ckpt['epoch'] + 1
    best_val_iou   = ckpt['best_val_iou']
    patience_count = ckpt['patience_count']
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            training_log = json.load(f)
    print(f'Resumed from epoch {start_epoch}, best IoU: {best_val_iou:.4f}')
else:
    print('Starting fresh training.')


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss  = 0
    all_metrics = {'iou': 0, 'f1': 0, 'precision': 0, 'recall': 0}
    n_batches   = 0
    desc = 'Train' if training else 'Val'
    ctx  = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        pbar = tqdm(loader, desc=desc, leave=False)
        for spatial, temporal, label in pbar:
            spatial  = spatial.to(DEVICE,  non_blocking=True)
            temporal = temporal.to(DEVICE, non_blocking=True)
            label    = label.to(DEVICE,    non_blocking=True)

            if (label != -1).sum() == 0:
                continue

            pred = model(spatial, temporal)
            loss = combined_loss(pred, label)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            total_loss += loss.item()
            metrics     = compute_metrics(pred, label)
            for k in all_metrics:
                all_metrics[k] += metrics[k]
            n_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'iou': f'{metrics["iou"]:.4f}'})

    avg = {k: v / n_batches for k, v in all_metrics.items()}
    avg['loss'] = total_loss / n_batches
    return avg


print(f'Starting from epoch {start_epoch} to {MAX_EPOCHS}')
print('-' * 90)

for epoch in range(start_epoch, MAX_EPOCHS):
    t0 = time.time()
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics   = run_epoch(val_loader,   training=False)
    scheduler.step(val_metrics['loss'])
    elapsed = time.time() - t0

    epoch_log = {
        'epoch': epoch,
        'train': {k: round(v, 4) for k, v in train_metrics.items()},
        'val'  : {k: round(v, 4) for k, v in val_metrics.items()},
        'lr'   : optimizer.param_groups[0]['lr'],
        'time_s': round(elapsed, 1),
    }
    training_log.append(epoch_log)

    print(
        f'Ep {epoch:>3}/{MAX_EPOCHS} | '
        f'Train loss={train_metrics["loss"]:.4f} IoU={train_metrics["iou"]:.4f} | '
        f'Val loss={val_metrics["loss"]:.4f} IoU={val_metrics["iou"]:.4f} '
        f'F1={val_metrics["f1"]:.4f} | '
        f'LR={optimizer.param_groups[0]["lr"]:.1e} | {elapsed:.0f}s'
    )

    torch.save({
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_iou': best_val_iou, 'patience_count': patience_count,
    }, LAST_MODEL_PATH)

    if val_metrics['iou'] > best_val_iou:
        best_val_iou   = val_metrics['iou']
        patience_count = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'  ✅ New best val IoU: {best_val_iou:.4f}')
    else:
        patience_count += 1
        print(f'  No improvement ({patience_count}/{PATIENCE})')

    with open(LOG_PATH, 'w') as f:
        json.dump(training_log, f, indent=2)

    if patience_count >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nTraining complete. Best val IoU: {best_val_iou:.4f}')
with open(f'{CKPT_DIR}/final_metrics.json', 'w') as f:
    json.dump({'best_val_iou': best_val_iou, 'epochs_trained': epoch + 1}, f, indent=2)

# 6 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

with open(LOG_PATH) as f:
    log = json.load(f)

epochs     = [e['epoch'] for e in log]
train_loss = [e['train']['loss'] for e in log]
val_loss   = [e['val']['loss']   for e in log]
train_iou  = [e['train']['iou']  for e in log]
val_iou    = [e['val']['iou']    for e in log]
val_f1     = [e['val']['f1']     for e in log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_loss, label='Train loss')
ax1.plot(epochs, val_loss,   label='Val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss — U-Net + ConvLSTM V2'); ax1.legend(); ax1.grid(True)

ax2.plot(epochs, train_iou, label='Train IoU')
ax2.plot(epochs, val_iou,   label='Val IoU')
ax2.plot(epochs, val_f1,    label='Val F1', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score')
ax2.set_title('IoU & F1 — U-Net + ConvLSTM V2'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# 7 — Final Evaluation

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()
val_metrics = run_epoch(val_loader, training=False)

print('=== Final Val Metrics — U-Net + ConvLSTM V2 ===')
print(f'  IoU       : {val_metrics["iou"]:.4f}')
print(f'  F1 / Dice : {val_metrics["f1"]:.4f}')
print(f'  Precision : {val_metrics["precision"]:.4f}')
print(f'  Recall    : {val_metrics["recall"]:.4f}')
print(f'  Loss      : {val_metrics["loss"]:.4f}')

print('\n=== Full Ablation Table ===')
print(f'{"Variant":40s}  {"IoU":>7}  {"F1":>7}')
print('-' * 58)
print(f'{"1 — U-Net spatial only":40s}  {0.7315:>7.4f}  {0.8245:>7.4f}')
print(f'{"2 — U-Net + ConvLSTM (hidden=128, 1L)": 40s}  {0.7315:>7.4f}  {0.8254:>7.4f}')
print(f'{"3 — U-Net + LSTM":40s}  {"TBD":>7}  {"TBD":>7}')
print(f'{"4 — U-Net + ConvLSTM (hidden=256, 2L, attn)":40s}  {val_metrics["iou"]:>7.4f}  {val_metrics["f1"]:>7.4f}')

with open(f'{CKPT_DIR}/final_metrics.json', 'w') as f:
    json.dump(val_metrics, f, indent=2)

# 8 — Save to Kaggle

In [ ]:
USERNAME = 'majorproject777'
meta = {
    'title'   : 'flood-convlstm-v2-model2',
    'id'      : f'{USERNAME}/flood-convlstm-v2-model',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(f'{CKPT_DIR}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

!kaggle datasets create -p {CKPT_DIR} --dir-mode zip
print('Model uploaded to Kaggle ✅')